# Mini-Project: [Your Title Here]
**Intern:** 김규성  
**Date started:** 2026-07-28  


---
## Context

You have already worked with this dataset in Module 4:

- **Data:** Fur ChIP-exo binding sites from Seo et al. 2014 — *Nature Communications* 5:4910 (GEO series **GSE54901** for the raw signal; the discrete binding sites come from the paper's **Supplementary Data** table you downloaded in Module 4)
- **Organism:** *E. coli* K-12 MG1655
- **Files in `data/reference/`:** Fur binding-site table (iron-replete sites, with `ChIP-exo Start`/`End`, `S/N ratio`, `Significance`, `Distance to TSS`), `NC_000913.fasta`, `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`, and the `fur_sites_for_meme.fasta` you generated.

Module 4 asked one specific question: *find the FUR binding motif.* This project asks you to go further and answer a question of your own choosing about the FUR regulon.


---
## Schedule Overview

| Day | Target milestone |
|-----|------------------|
| W5 D1 | Literature review done; candidate questions drafted |
| W5 D2 | Biological question finalized and instructor-approved |
| W5 D3 | Analysis plan written; pipeline scaffold annotated |
| W5 D4–5 | Pipeline running; first results visible |
| W6 D1–3 | All steps done; figures generated |
| W6 D4–5 | Report written; notebook submitted |

Run `/log` at the end of **every day**.


---
## Part 1 — What Else Is Known About FUR?
**Target: W5 Day 1**

You already know the Seo et al. 2014 dataset. Before proposing a question, find out what other groups have studied about the FUR regulon in *E. coli* — you want to ask something that either extends, challenges, or fills a gap in the existing literature.

Use Claude to help search and summarize. Key questions to answer:
- What genes are in the known FUR regulon?
- What is already known about FUR binding site position relative to promoters/TSS?
- Are there known differences between iron-replete and iron-depleted conditions?
- What aspects of FUR binding have *not* been studied with ChIP-exo?


**Literature summary** — 4–6 sentences covering what is known and what gaps exist:


> **Answer**
>
> FUR Regulon에는 철 흡수와 관련된 enterobactin(entCEBA, ybdB, entD 등), ferrichrome/hydroxamate siderophore system(fhuA, fhuB, fhuC 등), Ferric citrate system(fecABCDE, fecIR) 등 철의 흡수와 저장 및 스트레스 대응 유전자들이 포함된다.
>
> FUR binding site와 promoter/TSS의 위치 관계가 알려져 있다 - Fur box: 19 bp의 palindromic 서열 "GATAATGATAATCATTATC"이 알려져 있으며, Baichoo & Helmann, 2002에서 실제 Fur은 7-1-7 bp의 inverted repeat 서열을 인식하는 것으로 알려져 있다.
>
> 철이 충분한 환경에서 Fur는 Fe2+가 결합한 holo 형태로 존재하여, dimer 및 polymer를 이루며 유전자의 RNAP-binding region 바로 위 또는 근처의 Fur box에 결합, RNAP 또는 시그마 인자가 접근하는 것을 물리적으로 차단한다. -> holo-Fur repression
>
> 철이 부족한 환경에서는 Fur가 DNA 친화성이 낮은 apo 형태로 존재하여 genome에서 이탈하게 된다. -> derepression, a.k.a. apo-Fur activation
>
> ChIP-exo로 아직 연구되지 않은 부분: (1) 현재 데이터는 mid-exponential 시기에 ChIP-exo를 함. exponential~stationary transition 시기에 같은 조건으로 ChIP-exo 연구된 바 없음. (2) iron replete vs chelator 첨가 후 단일 고정된 timepoint에서만 측정함. 철이 고갈됨에 따라 Fur가 binding site를 떠나는 역학은 연구되지 않음. (3) Fur oligomeric stoichiometry: ChIP-exo는 binding-footprint의 위치와 대략적인 signal strength만 제공하는 한계가 있어, 특정 tandem/polymerized site에 몇 개의 Fur dimer가 결합하는지, 또는 결합되는 Fur의 assembly의 순서는 알 수 없음. single-molecule imaging, AFM 등 orthogonal biophysical method가 필요. (4) Fur는 OxyR(산화 스트레스 반응), acid stress, biofilm regulator들과 cross-talk하는 것으로 알려져 있으나, 철 유무를 산화스트레스/pH 스트레스/삼투 스트레스와 동시에 변화시켜 condition-dependent co-occupancy shift를 mapping한 ChIP-exo 데이터셋은 없음 (5) single-cell heterogeneity: ChIP-exo는 population-averaged된 방법. Fur occupancy가 population 전체에서 균일한지, 혹은 biomdal/heterogenous한지는 ChIP-exo 만으로 알수 없음. (6) 동일 실험 프레임워크 내에서 RyhB/Hfq와의 direct co-mapping이 되어 있지 않음. RyhB-mediated된 regulon의 간접적인 arm은 유전학적으로, 그리고 RNA-seq을 통해 잘 규명되어 있음. 그러나 이것이 동일한 ChIP-exo binding map과 통합된 단일 연구로 직접 연결된 적이 없어, RyhB 자신의 타겟들은 Fur occupancy와 함께 co-profiling되지 않고, 별도의 sRNA-target 데이터셋으로부터 추론됨.
>
> (3)에서 착안하여, 본 프로젝트는 두 가설을 검증한다. 여기서 'tandem Fur box'는 하나의 Fur box 내부의 GATAAT 반복(Baichoo & Helmann 2002)이 아니라, Seo et al. 2014의 `# of motifs` 컬럼이 나타내는 — 한 결합 부위 안에 완전한 Fur box가 여러 개(n=1–4) 존재하는 경우를 가리킨다. (가설 1) n이 클수록 결합 친화도가 높아져 ChIP-exo S/N ratio도 함께 증가할 것이다. (가설 2) 반면 전사 억제는 holo-Fur dimer 하나만으로도 RNAP/시그마 인자 결합 부위를 충분히 가릴 수 있는 steric-hindrance 기전이므로, n이 늘어도 발현 억제 정도(iron-replete vs iron-starve log2FC)는 어느 지점 이후 포화되어 occupancy 증가만큼 비례해 강해지지는 않을 것이다.


> **W5 D1 checkpoint:**
> - [ ] At least 2 papers read beyond Seo et al. 2014
> - [ ] Gap or open question identified
> - [ ] `/log` run


---
## What Makes a Good Biological Question?

Before writing your question, use this checklist:

| Criterion | Ask yourself |
|-----------|-------------|
| **Answerable with available data** | Can you answer it using the files already in `data/reference/`? |
| **Not already answered** | Did Seo et al. 2014 or the papers you just read already answer this? |
| **Has a plausible mechanism** | If you found a pattern, would there be a biological reason for it? |
| **Specific and testable** | Can you describe the analysis in 2–3 steps and know what result would count as a "yes" or "no"? |

> **What the binding-site table gives you:** each Fur site has a `ChIP-exo Start`/`End`, a **`S/N ratio`** and a **`Significance (p-value)`** (these are the *signal-strength* / *score* fields — the `Peak` column is just an ID label like `P1`), a `Binding Condition` (`R` iron-replete / `S` iron-depleted / `R/S` both), and a `Distance to TSS` for many sites. Design your question around columns that actually exist.

### Examples of tractable questions with the FUR dataset

- *Do FUR binding sites closer to a TSS have a higher S/N ratio than sites far from a TSS?*
- *Are FUR sites that bind in both conditions (`R/S`) stronger (higher S/N ratio) than condition-specific ones?*
- *Does the GC content of FUR binding site sequences correlate with the site's S/N ratio?*
- *Which functional categories (metabolism, transport, stress response) of the target genes are most enriched among FUR binding sites?*

### Examples of questions to avoid

- *"What does FUR do?"* — already answered in the literature
- *"How many peaks are there?"* — answered in Module 4
- *"Is FUR important for iron regulation?"* — too broad, not answerable with this dataset alone

### Where to search for background

- **PubMed:** `pubmed.ncbi.nlm.nih.gov` — search `"FUR regulon E. coli"` or `"Fur binding E. coli iron"`
- **Google Scholar:** same search terms
- **The Seo et al. 2014 paper itself** — its Discussion section lists open questions

---


---
## Part 2 — Biological Question
**Target: W5 Day 2**

Propose **one specific, answerable** question about the FUR ChIP-exo dataset that goes beyond what Module 4 already showed you.

The question must be answerable using the files in `data/reference/`. If you need additional tools or datasets, use Claude Code to set them up.

Example questions at the right level of specificity:
- *"Do FUR binding sites with a higher S/N ratio show stronger motif matches than low-signal sites?"*
- *"Which genes in the lab annotation have a FUR binding site within their upstream region, and can you find a pattern in what those genes do?"*
- *"Do iron-replete–specific FUR sites (`R`) differ in location (regulatory vs internal) from sites bound in both conditions (`R/S`)?"*
- *"How does the GC content of FUR binding site sequences compare to randomly sampled genomic windows of the same length — and does GC content correlate with S/N ratio?"*

**Not acceptable:** *"What does FUR do?"* or *"How many peaks are there?"* (already answered in M4)


**My biological question:**


> **Answer**
>
> Fur ChIP-exo 결합 부위 내 tandem Fur box 개수(Seo et al. 2014의 `# of motifs`, n=1–4)가 증가할수록 ChIP-exo 결합 신호(S/N ratio)는 계속 비례해서 증가하는가, 반면 그 부위가 조절하는 유전자의 실제 전사 억제 강도(iron-replete vs iron-starved RNA-seq log2FC)는 n이 일정 수준을 넘으면 포화(saturate)되어 occupancy만큼 비례해서 강해지지 않는가?


> **INSTRUCTOR CHECK-IN REQUIRED** before proceeding to Part 3.  
> Do not start the analysis plan until your question is approved.


---
## Part 3 — Analysis Plan
**Target: W5 Day 3**

Write your analysis plan **before** opening Claude Code. This is your own thinking.

Your plan must answer:
1. Which files in `data/reference/` will you use?
2. What steps will you run, in order?
3. What output do you expect at each step?
4. How will you decide if the result is biologically meaningful?


> **Answer**
>
> [Your analysis plan here — write this before asking Claude Code]


**Pipeline scaffold** — use plan mode (Shift+Tab twice) to generate the pipeline.  
Review every proposed step. Annotate each command with your own understanding of what it does.


In [ ]:
# Pipeline scaffold — generated with plan mode, annotated by you.
# If any cell holds shell commands (bowtie2, samtools, fastq-dump, meme...),
# make its FIRST line %%bash — otherwise Jupyter runs them as Python and errors.
# Pure-Python analysis cells (pandas, Biopython, matplotlib) need no magic.



> **W5 D3 checkpoint:**
> - [ ] Analysis plan written in your own words
> - [ ] Pipeline scaffold annotated (you can explain every command)
> - [ ] Instructor has approved your biological question
> - [ ] `/log` run


---
## Part 4 — Analysis Execution
**Target: W5 Days 4–5 and W6 Days 1–3**

Run your pipeline step by step. Add code cells as needed.  
When blocked: paste the error + command + relevant output into Claude Code and ask for diagnosis — then use `/debug`.

Track your progress:

| Step | Status |
|------|--------|
| [Step 1 name] | not started / in progress / done |
| [Step 2 name] | not started / in progress / done |
| [Step 3 name] | not started / in progress / done |



In [ ]:
# Step 1


> **Answer**
>
> *What did this step produce? Any issues?*


In [ ]:
# Step 2


> **Answer**
>
> *What did this step produce? Any issues?*


In [ ]:
# Step 3


> **Answer**
>
> *What did this step produce? Any issues?*


*(Add more step cells as needed.)*


> **W5 D5 checkpoint:**
> - [ ] At least first 2 steps producing output
> - [ ] Any blockers documented in `/log`
> - [ ] `/log` run


---
## Part 5 — Results
**Target: W6 Days 1–3**

Document your results. Each figure needs a 1–2 sentence caption — what it shows and what you observe.


In [ ]:
# Figure 1


> **Answer**
>
> **Figure 1:** [What this figure shows and what you observe]


In [ ]:
# Figure 2 (add as needed)


> **Answer**
>
> **Figure 2:** [Caption]


> **W6 D3 checkpoint:**
> - [ ] All pipeline steps complete
> - [ ] At least one figure with a written caption
> - [ ] `/log` run


---
## Part 6 — Report
**Target: W6 Days 4–5**

Write your report in the cells below. This is **your own writing** — Claude may be consulted for phrasing, but the scientific content must be yours.


### Question
*(one sentence)*


> **Answer**
>
> [Your question here]


### Methods
*(tools, parameters, files used — enough to reproduce)*


> **Answer**
>
> [Your methods here]


### Results
*(what did you find? reference figures by number)*


> **Answer**
>
> [Your results here]


### Interpretation
*(what does this mean biologically? how does it compare to the known FUR regulon?)*


> **Answer**
>
> [Your interpretation here]


### Limitations
*(what would you do differently with more time?)*


> **Answer**
>
> [Your limitations here]


---
Run `/log` one last time, then submit this notebook as your deliverable.


## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(miniproject): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
